# 4. Word2Vec-style embeddings

This notebook learns small **Word2Vec-style embeddings** from `observations.csv`.

The fictional-language tokens are treated as words. We build target/context pairs from a symmetric context window and learn vectors using a skip-gram-like **logistic objective with negative sampling**.

We compare:

- window **2**: one token before + one token after;
- window **4**: two tokens before + two tokens after;
- embedding dimensions **2** and **3**.

The resulting vectors are plotted as lines from the origin to each token.

> This is a pedagogical implementation. It exposes the vectors and the logistic objective directly rather than using a Word2Vec library.


In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Configuration
INPUT_FILE = "../01-observations/observations.csv"
WINDOW_SIZES = [2, 4]
EMBEDDING_DIMS = [2, 3]
NEGATIVE_SAMPLES_PER_POSITIVE = 3
EPOCHS = 800
LEARNING_RATE = 0.05
RANDOM_SEED = 42

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)


In [2]:
# Read observations
df = pd.read_csv(INPUT_FILE)
sentences = [s.split() for s in df["sentence"]]

vocabulary = sorted(set(t for s in sentences for t in s))
token_to_id = {t: i for i, t in enumerate(vocabulary)}

print(f"Observations: {len(sentences)}")
print(f"Vocabulary: {vocabulary}")
print(f"Vocabulary size: {len(vocabulary)}")


Observations: 850
Vocabulary: ['e', 'f', 'l', 'p', 'r', 's']
Vocabulary size: 6


## Build target/context pairs

With window 2, a target sees one token on either side.

With window 4, a target sees up to two tokens on either side.

These are the positive examples for the skip-gram objective.


In [3]:
def build_positive_pairs(sequences, window_size):
    radius = window_size // 2
    pairs = []
    for sentence in sequences:
        for i, target in enumerate(sentence):
            lo = max(0, i - radius)
            hi = min(len(sentence), i + radius + 1)
            for j in range(lo, hi):
                if i != j:
                    pairs.append((target, sentence[j]))
    return pairs

for w in WINDOW_SIZES:
    pairs = build_positive_pairs(sentences, w)
    print(f"Window {w}: {len(pairs)} positive pairs")


Window 2: 6850 positive pairs
Window 4: 12000 positive pairs


## Logistic-regression view of Word2Vec

For target vector \(v_w\) and context vector \(u_c\), the pair score is

\[
score(w,c)=v_w \cdot u_c.
\]

The logistic probability is

\[
P(y=1|w,c)=\sigma(v_w\cdot u_c).
\]

Observed target/context pairs have \(y=1\); randomly sampled pairs have \(y=0\).

We optimize the binary cross-entropy directly. This gives a compact implementation of the central Word2Vec idea while keeping the learned embedding vectors visible.


In [4]:
def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1 / (1 + np.exp(-x))

def train_embeddings(sequences, window_size, dim, epochs=EPOCHS,
                     learning_rate=LEARNING_RATE,
                     negative_ratio=NEGATIVE_SAMPLES_PER_POSITIVE,
                     seed=42):
    rng = np.random.default_rng(seed)
    V = len(vocabulary)

    # Separate target and context vectors, as in skip-gram.
    W = rng.normal(0, 0.15, (V, dim))
    U = rng.normal(0, 0.15, (V, dim))

    positive = build_positive_pairs(sequences, window_size)
    counts = Counter(positive)
    positive_types = list(counts)

    # Generate negative pairs.
    negative = []
    positive_set = set(positive_types)
    for target, _ in positive_types:
        for _ in range(negative_ratio):
            while True:
                context = rng.choice(vocabulary)
                if (target, context) not in positive_set:
                    negative.append((target, context))
                    break

    # Repeat positive types according to their observed counts.
    training = [(p, 1) for p in positive for _ in [0]]
    training += [(p, 0) for p in negative]

    # Shuffle and train with stochastic gradient descent.
    losses = []

    for epoch in range(epochs):
        rng.shuffle(training)
        total_loss = 0.0

        for (target, context), y in training:
            wi = token_to_id[target]
            ci = token_to_id[context]

            v = W[wi].copy()
            u = U[ci].copy()

            score = np.dot(v, u)
            prob = sigmoid(score)
            error = prob - y

            W[wi] -= learning_rate * error * u
            U[ci] -= learning_rate * error * v

            total_loss += -(y*np.log(prob+1e-12) +
                            (1-y)*np.log(1-prob+1e-12))

        if epoch % max(1, epochs//10) == 0:
            losses.append(total_loss / len(training))

    embeddings = pd.DataFrame(W, index=vocabulary)
    embeddings.index.name = "token"
    return embeddings, losses


## Train the four configurations

We train:

- window 2 × dimension 2
- window 2 × dimension 3
- window 4 × dimension 2
- window 4 × dimension 3


In [5]:
models = {}

for window in WINDOW_SIZES:
    for dim in EMBEDDING_DIMS:
        print(f"Training window={window}, dimension={dim}")
        emb, losses = train_embeddings(
            sentences, window, dim,
            seed=(RANDOM_SEED or 0) + window*100 + dim
        )
        models[(window, dim)] = {"embeddings": emb, "losses": losses}
        display(emb.round(3))


Training window=2, dimension=2


KeyboardInterrupt: 

## 2D embeddings

Each line starts at the origin and ends at the learned vector for one token.


In [ ]:
def plot_2d(embeddings, title):
    fig, ax = plt.subplots(figsize=(8, 7))
    for token, row in embeddings.iterrows():
        x, y = row.iloc[0], row.iloc[1]
        ax.plot([0, x], [0, y], linewidth=1)
        ax.scatter([x], [y], s=45)
        ax.text(x, y, f"  {token}", fontsize=11, va="center")
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("Embedding dimension 1")
    ax.set_ylabel("Embedding dimension 2")
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.show()

for window in WINDOW_SIZES:
    plot_2d(models[(window, 2)]["embeddings"],
            f"Word2Vec-style embeddings — window {window}, dimension 2")


## 3D embeddings

Again, each line starts at the origin and ends at the token's learned vector.


In [ ]:
def plot_3d(embeddings, title):
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    for token, row in embeddings.iterrows():
        x, y, z = row.iloc[0], row.iloc[1], row.iloc[2]
        ax.plot([0, x], [0, y], [0, z], linewidth=1)
        ax.scatter([x], [y], [z], s=45)
        ax.text(x, y, z, f"  {token}", fontsize=10)
    ax.set_xlabel("Embedding dimension 1")
    ax.set_ylabel("Embedding dimension 2")
    ax.set_zlabel("Embedding dimension 3")
    ax.set_title(title)
    plt.show()

for window in WINDOW_SIZES:
    plot_3d(models[(window, 3)]["embeddings"],
            f"Word2Vec-style embeddings — window {window}, dimension 3")


## Cosine similarity

The embedding vectors can also be compared by cosine similarity.


In [ ]:
def cosine_similarity_matrix(embeddings):
    X = embeddings.to_numpy()
    X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return pd.DataFrame(X @ X.T, index=embeddings.index, columns=embeddings.index)

for (window, dim), model in models.items():
    print(f"Window={window}, dimension={dim}")
    display(cosine_similarity_matrix(model["embeddings"]).round(3))


In [ ]:
# ------------------------------------------------------------
# Save embeddings and cosine similarities
# ------------------------------------------------------------
#
# One pair of CSV files is produced for each configuration.
# The names encode the context window and embedding dimension.

def save_embedding_results(models):
    for (window, dim), model in models.items():
        embeddings = model["embeddings"]

        embedding_file = f"embeddings-{window}-gram-{dim}d-matrix.csv"
        similarity_file = f"embeddings-{window}-gram-{dim}d-cosine-similarity.csv"

        # Embedding matrix: one row per token, one column per dimension.
        embeddings.to_csv(embedding_file)

        # Cosine similarity matrix: tokens in both rows and columns.
        similarity = cosine_similarity_matrix(embeddings)
        similarity.to_csv(similarity_file)

        print(f"Saved: {embedding_file}")
        print(f"Saved: {similarity_file}")


save_embedding_results(models)
